# Generating ESM-2 Embeddings 

ESM-2 sequence sequence is embedded via mean-pooling over the final hidden layer of
ESM-2 (esm2_t6_8M_UR50D, 320-dim). Results are appended as a new column
`esm_vector` and saved to CSV.

Authentication: run `huggingface-cli login` before executing this script,
or set the HF_TOKEN environment variable.'''

In [1]:
import os
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import EsmTokenizer, EsmModel
from huggingface_hub import login

/Users/matteo/Desktop/Uttopia_all_shit/ALL_CODE_/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup & Configuration

In [2]:
INPUT_FILE  = "/Users/matteo/Desktop/Uttopia_all_shit/ALL_CODE_/A_Dataset_Human/1_All_protein_uniprot_human.csv"
OUTPUT_FILE = "/Users/matteo/Desktop/Uttopia_all_shit/ALL_CODE_/A_Dataset_Human/2_All_protein_esm_humann.csv"

MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

# Available variants and their embedding dimensions:
#   esm2_t6_8M_UR50D    →  320
#   esm2_t12_35M_UR50D  →  480
#   esm2_t30_150M_UR50D →  640
#   esm2_t33_650M_UR50D →  1280

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

###  HuggingFace Authentication

In [3]:
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)

### Model Loading

In [4]:
tokenizer = EsmTokenizer.from_pretrained(MODEL_NAME)
model = EsmModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

Loading weights: 100%|██████████| 107/107 [00:00<00:00, 7985.03it/s]
EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 320, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-5): 6 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=320, out_features=320, bias=True)
            (key): Linear(in_features=320, out_features=320, bias=True)
            (value): Linear(in_features=320, out_features=320, bias=True)
            (rotary_embeddings): RotaryEmbedding()
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=320, out_features=320, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=320, out_features=1280, bias=True)
        )
        (output): EsmOutput(
        

### Embedding Function

In [5]:
def get_esm_embedding(sequence: str) -> np.ndarray | None:
    """
    Returns the mean-pooled ESM-2 embedding for a protein sequence.
    Sequences longer than 1024 residues are truncated to fit the model's
    context window. Returns None for missing or invalid entries.
    """
    if pd.isna(sequence) or sequence == "NOT_FOUND" or not sequence:
        return None

    inputs = tokenizer(
        sequence[:1024],
        return_tensors = "pt",
        padding = True,
        truncation = True,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy().flatten()

    return embedding

### Embedding Generation

In [6]:
df = pd.read_csv(INPUT_FILE)
esm_vectors = []

for _, row in tqdm(df.iterrows(), total = len(df), desc = "Generating embeddings"):
    v = get_esm_embedding(row["seq_prot"])
    esm_vectors.append(v.tolist() if v is not None else None)

df["esm_vector"] = esm_vectors
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved → {OUTPUT_FILE} :)")

Generating embeddings: 100%|██████████| 1/1 [00:00<00:00,  4.00it/s]

Saved → /Users/matteo/Desktop/Uttopia_all_shit/ALL_CODE_/A_Dataset_Human/2_All_protein_esm_humann.csv :)
